## Load Data

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_excel("Football_Dataset.xlsx")

df.head()

,PlayerID,PlayerName,Team,Position,MatchDate,Opponent,MinutesPlayed,Goals,Assists,Shots,...,KeyPasses,Dribbles,PassAccuracy,Tackles,Interceptions,Clearances,FoulsCommitted,YellowCards,RedCards,Rating
0,4591,Player_38,Brighton,Forward,2021-08-18,Aston Villa,80,3,1,6,...,3,2,68,1,3,1,2,0,1,6.1
1,3091,Player_10,Aston Villa,Defender,2022-01-12,Chelsea,86,3,1,4,...,4,4,87,0,2,4,1,1,1,8.2
2,9023,Player_27,Aston Villa,Defender,2021-10-13,Manchester City,69,2,0,4,...,4,1,73,5,3,7,3,0,1,6.2
3,9899,Player_86,Newcastle,Forward,2022-04-24,Crystal Palace,69,3,1,7,...,5,4,80,2,5,4,3,2,1,7.6
4,5506,Player_113,Leeds United,Midfielder,2022-01-18,Brighton,80,2,2,2,...,1,4,91,5,1,6,2,1,0,7.4


## Data Cleaning and Preprocessing

In [2]:
# Check data types and missing values
df.info()

# Convert MatchDate to datetime objects
df["MatchDate"] = pd.to_datetime(df["MatchDate"])

# Remove rows with missing values and eliminate duplicate entries
df = df.dropna().drop_duplicates()

# Summary of null values after cleaning
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 22 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PlayerID        5000 non-null   int64  
 1   PlayerName      5000 non-null   object 
 2   Team            5000 non-null   object 
 3   Position        5000 non-null   object 
 4   MatchDate       5000 non-null   object 
 5   Opponent        5000 non-null   object 
 6   MinutesPlayed   5000 non-null   int64  
 7   Goals           5000 non-null   int64  
 8   Assists         5000 non-null   int64  
 9   Shots           5000 non-null   int64  
 10  xG              5000 non-null   float64
 11  xA              5000 non-null   float64
 12  KeyPasses       5000 non-null   int64  
 13  Dribbles        5000 non-null   int64  
 14  PassAccuracy    5000 non-null   int64  
 15  Tackles         5000 non-null   int64  
 16  Interceptions   5000 non-null   int64  
 17  Clearances      5000 non-null   i

## Main Data Aggregation

In [26]:
# Aggregate raw metrics for all sections
player_stats = df.groupby("PlayerName").agg({
    "Goals": "sum", "Assists": "sum", "Shots": "sum", "xG": "sum", "xA": "sum",
    "KeyPasses": "sum", "Dribbles": "sum", "PassAccuracy": "mean", "Tackles": "sum",
    "Interceptions": "sum", "Clearances": "sum", "FoulsCommitted": "sum",
    "YellowCards": "sum", "RedCards": "sum", "Rating": "mean",
    "MinutesPlayed": "sum", "MatchDate": "nunique", "Team": "first"
}).reset_index()

player_stats.rename(columns={"MatchDate": "Matches", "Rating": "Avg_Rating"}, inplace=True)

# Global Team Metrics
team_metrics = df.groupby("Team").agg({"Goals": "sum", "KeyPasses": "sum", "MinutesPlayed": "sum"}).reset_index()
team_metrics.columns = ["Team", "Team_Total_Goals", "Team_Total_KP", "Team_Total_Minutes"]
player_stats = player_stats.merge(team_metrics, on="Team", how="left")

# Helper Counts (Matches with actions)
matches_with_goals = df[df["Goals"] > 0].groupby("PlayerName")["MatchDate"].nunique().reset_index()
matches_with_goals.columns = ["PlayerName", "Matches_With_Goals"]

gc_matches = df[(df["Goals"] > 0) | (df["Assists"] > 0)].groupby("PlayerName")["MatchDate"].nunique().reset_index()
gc_matches.columns = ["PlayerName", "GC_Matches"]

creative_matches = df[df["KeyPasses"] > 0].groupby("PlayerName")["MatchDate"].nunique().reset_index()
creative_matches.columns = ["PlayerName", "Creative_Matches"]

# Variance & Standard Deviations
stds = df.groupby("PlayerName").agg({"Goals": "var", "PassAccuracy": "std", "Tackles": "std", "MinutesPlayed": "std"}).reset_index()
stds.columns = ["PlayerName", "Goals_Variance", "Pass_Std", "Tackle_Std", "Minutes_Std"]

# Merge helpers
player_stats = player_stats.merge(matches_with_goals, on="PlayerName", how="left")
player_stats = player_stats.merge(gc_matches, on="PlayerName", how="left")
player_stats = player_stats.merge(creative_matches, on="PlayerName", how="left")
player_stats = player_stats.merge(stds, on="PlayerName", how="left")
player_stats.fillna(0, inplace=True)

A. Goal Scoring KPIs

In [27]:
# Average goals per match and per 90 minutes played
player_stats["Goals_per_Match"] = player_stats["Goals"] / player_stats["Matches"]
player_stats["Goals_per_90"] = (player_stats["Goals"] / player_stats["MinutesPlayed"]) * 90

# Shot quality and finishing accuracy
player_stats["Shot_Conversion"] = player_stats["Goals"] / (player_stats["Shots"] + 0.1)
player_stats["xG_Efficiency"] = player_stats["Goals"] - player_stats["xG"]
player_stats["Finishing_Efficiency"] = player_stats["Goals"] / (player_stats["xG"] + 0.1)

# Team reliance and scoring stability
player_stats["Goal_Participation"] = player_stats["Goals"] / (player_stats["Team_Total_Goals"] + 0.1)
player_stats["Goal_Consistency"] = player_stats["Matches_With_Goals"] / player_stats["Matches"]
player_stats["Finishing_vs_Volume"] = player_stats["Goals_per_90"] / ((player_stats["Shots"] / player_stats["MinutesPlayed"]) * 90 + 0.1)

B. Goal Contribution KPIs

In [28]:
# Total direct involvements (Goals + Assists) and their rates
player_stats["Goal_Contribution"] = player_stats["Goals"] + player_stats["Assists"]
player_stats["GC_per_Match"] = player_stats["Goal_Contribution"] / player_stats["Matches"]
player_stats["GC_per_90"] = (player_stats["Goal_Contribution"] / player_stats["MinutesPlayed"]) * 90

# Efficiency of actual assists relative to expected assists (xA)
player_stats["Assist_Efficiency"] = player_stats["Assists"] / (player_stats["xA"] + 0.1)

# Direct contribution share of the team's total goals
player_stats["GC_Percentage"] = player_stats["Goal_Contribution"] / (player_stats["Team_Total_Goals"] + 0.1)

# Weighted score for overall attacking impact
player_stats["Creative_Impact_B"] = (player_stats["Goals"] * 0.6) + (player_stats["Assists"] * 0.8) + (player_stats["xA"] * 0.4)

# Frequency of goal contribution across matches
player_stats["Contribution_Consistency"] = player_stats["GC_Matches"] / player_stats["Matches"]

C. Shooting Quality KPIs

In [29]:
# Quality of chances selected (Average xG per shot)
player_stats["xG_per_Shot"] = player_stats["xG"] / (player_stats["Shots"] + 0.1)

# Direct output per shot (Goals + Assists per shot taken)
player_stats["Shot_Productivity"] = (player_stats["Goals"] + player_stats["Assists"]) / (player_stats["Shots"] + 0.1)

# Composite score for finishing intelligence and shot value
player_stats["Finishing_Intelligence"] = (
    (player_stats["Goals"] / (player_stats["Shots"] + 0.1)) * 0.5 + 
    (player_stats["Goals"] / (player_stats["xG"] + 0.1)) * 0.3 + 
    (player_stats["xG_per_Shot"] * 0.2)
)

# Total weighted shooting impact
player_stats["Shooting_Impact"] = (
    (player_stats["Goals"] * 0.6) + 
    ((player_stats["Goals"] / (player_stats["Shots"] + 0.1)) * 0.2) + 
    (player_stats["Shot_Productivity"] * 0.2)
)

D. Creativity & Playmaking KPIs

In [30]:
# Key passes frequency and dribbling volume
player_stats["KeyPasses_per_90"] = (player_stats["KeyPasses"] / player_stats["MinutesPlayed"]) * 90
player_stats["Dribbles_per_Match"] = player_stats["Dribbles"] / player_stats["Matches"]
player_stats["Chance_Creation_Index"] = player_stats["KeyPasses"] + player_stats["Assists"]

# Ability to convert key passes into actual assists
player_stats["Creativity_Efficiency"] = player_stats["Assists"] / (player_stats["KeyPasses"] + 1)

# Player's share of team's total playmaking output
player_stats["Playmaking_Load"] = player_stats["KeyPasses"] / (player_stats["Team_Total_KP"] + 0.1)

# Consistency of creativity across all matches
player_stats["Creative_Consistency"] = player_stats["Creative_Matches"] / player_stats["Matches"]

# The impact of dribbling on creating clear chances
player_stats["Dribble_Impact"] = player_stats["Dribbles_per_Match"] * player_stats["Chance_Creation_Index"]

E. Passing Performance KPIs

In [31]:
# General accuracy and stability of passing across matches
player_stats["Avg_Pass_Accuracy"] = player_stats["PassAccuracy"]
player_stats["Passing_Consistency"] = player_stats["Avg_Pass_Accuracy"] / (player_stats["Pass_Std"] + 1)

# Weighted passing impact score
player_stats["Passing_Impact"] = (player_stats["Avg_Pass_Accuracy"] * 0.5) + (player_stats["Passing_Consistency"] * 0.5)

# Total possession value based on accuracy and volume
player_stats["Possession_Value"] = (player_stats["Avg_Pass_Accuracy"] * player_stats["MinutesPlayed"] / 100)

F. Defensive Performance KPIs

In [32]:
# Defensive actions volume per match and normalized per 90
player_stats["Tackles_per_Match"] = player_stats["Tackles"] / player_stats["Matches"]
player_stats["Defensive_Actions"] = player_stats["Tackles"] + player_stats["Interceptions"] + player_stats["Clearances"]
player_stats["Def_Actions_per_90"] = (player_stats["Defensive_Actions"] / player_stats["MinutesPlayed"]) * 90

# Overall defensive impact and activity intensity
player_stats["Defensive_Impact_F"] = (player_stats["Tackles"] * 0.4) + (player_stats["Interceptions"] * 0.3) + (player_stats["Clearances"] * 0.3)
player_stats["Defensive_Activity"] = player_stats["Def_Actions_per_90"] * (player_stats["Defensive_Actions"] / (player_stats["MinutesPlayed"] + 0.1))

G. Defensive Impact KPIs (Advanced)

In [33]:
# Measures the ability to win the ball without committing fouls
player_stats["Clean_Defense_Index"] = (player_stats["Tackles"] + player_stats["Interceptions"]) / (player_stats["FoulsCommitted"] + 1)

# Ratio of fouls to defensive interventions
player_stats["Aggression_Control_G"] = player_stats["FoulsCommitted"] / (player_stats["Tackles"] + player_stats["Interceptions"] + 1)

# General defensive discipline score
player_stats["Def_Discipline"] = 1 / (player_stats["FoulsCommitted"] + 1)

H. Discipline KPIs

In [34]:
# Card frequency and total discipline burden (Weighting cards)
player_stats["Card_Rate"] = (player_stats["YellowCards"] + player_stats["RedCards"]) / player_stats["Matches"]
player_stats["Discipline_Score"] = player_stats["FoulsCommitted"] + player_stats["YellowCards"] + (player_stats["RedCards"] * 2)

# Fouls committed for every card received
player_stats["Fouls_to_Cards"] = player_stats["FoulsCommitted"] / (player_stats["YellowCards"] + player_stats["RedCards"] + 1)

# Overall clean play indicator
player_stats["Clean_Play"] = 1 / (player_stats["Discipline_Score"] + 1)

I. Playing Time KPIs

In [35]:
# Share of team minutes played and match involvement
player_stats["Usage_Rate"] = player_stats["MinutesPlayed"] / (player_stats["Team_Total_Minutes"] + 0.1)
player_stats["Match_Involvement"] = player_stats["Matches"] / (player_stats["Matches"].max() + 0.1)

# Stability of playing time (Standard deviation check)
player_stats["Playing_Consistency"] = player_stats["MinutesPlayed"] / (player_stats["Minutes_Std"] + 1)

# General availability based on match appearance and minutes
player_stats["Availability_Score"] = player_stats["Matches"] * (player_stats["MinutesPlayed"] / (player_stats["Matches"] + 0.1))

J. Overall Performance KPIs

In [37]:
# Combined indexes for impact and overall contribution
player_stats["Performance_Index"] = player_stats["Avg_Rating"] + player_stats["Goals"] + player_stats["Assists"]
player_stats["Overall_Contribution"] = player_stats["Goals"] + player_stats["Assists"] + player_stats["KeyPasses"] + player_stats["Tackles"] + player_stats["Interceptions"]

# Weighted score for identifying the 'Star' impact
player_stats["Star_Index"] = player_stats["Avg_Rating"] * (player_stats["Goal_Contribution"] + 1)

# Final balanced score covering both offense and defense
player_stats["Balanced_Performance"] = ( (player_stats["Goal_Contribution"] * 0.5) + (player_stats["Defensive_Impact_F"] * 0.5) )

## Export Final Table

In [38]:
# Final cleanup and exporting the dataset
player_stats.replace([np.inf, -np.inf], 0, inplace=True)
player_stats.fillna(0, inplace=True)

# Save to excel for reporting
player_stats.to_excel("Football_Final_Analysis_Report.xlsx", index=False)
print("File Saved Successfully.")
player_stats.head()

File Saved Successfully.


,PlayerName,Goals,Assists,Shots,xG,xA,KeyPasses,Dribbles,PassAccuracy,Tackles,...,Fouls_to_Cards,Clean_Play,Usage_Rate,Match_Involvement,Playing_Consistency,Availability_Score,Performance_Index,Overall_Contribution,Star_Index,Balanced_Performance
0,Player_1,54,36,129,37.24,22.00,58,106,81.354839,95,...,1.600000,0.008621,0.126842,0.508167,206.232362,2323.701068,97.135484,318,649.329032,92.80
1,Player_10,62,26,129,34.53,23.05,75,76,76.909091,109,...,1.377358,0.006993,0.125439,0.598911,287.673671,2527.341390,95.351515,365,654.284848,98.65
2,Player_100,60,41,119,47.88,28.65,93,140,78.083333,88,...,1.245614,0.006757,0.141157,0.617060,256.342091,2686.099707,108.319444,374,746.583333,102.00
3,Player_101,54,29,121,46.86,25.99,86,110,81.187500,81,...,1.234043,0.008333,0.129373,0.562613,228.244617,2373.344051,90.337500,347,616.350000,93.25
4,Player_102,53,43,127,49.06,29.92,103,131,80.380952,132,...,1.338462,0.005882,0.156613,0.744102,284.911070,3157.299270,103.528571,436,730.271429,119.40
